# Inteligencia de Negocios — Primera lectura de datos crudos
## Proyecto: Análisis de hechos de tránsito en la Ciudad de México

**Equipo:** A&C  
**Evidencia para el Avance 1 — apartado 3.6**

### Pregunta de negocio
> ¿En qué alcaldías y horarios de la CDMX se concentra la mayor frecuencia de hechos de tránsito y en cuáles se observa una mayor severidad registrada, considerando las personas lesionadas y fallecidas?

### Cómo está pensado este notebook
Este archivo está preparado para **guardarse y consultarse desde el repositorio**, sin depender de una ruta C:\Users\....

Los CSV crudos son demasiado pesados para GitHub. Por eso:
1. el notebook conserva **las respuestas y resultados obtenidos de la primera lectura real** de los archivos que revisó el equipo;
2. incluye código reproducible para volver a calcularlos cuando los datos estén disponibles;
3. no realiza limpieza ni modifica los datos crudos;
4. los archivos crudos deben conservarse externamente y el repositorio debe documentar dónde descargarlos.

Así, la profesora puede abrir el notebook y leer la evidencia aunque los CSV grandes no estén almacenados en GitHub.

## 1. Inventario general de archivos revisados

La primera lectura se realizó sobre los archivos crudos de SSC y C5 proporcionados al equipo. INEGI-ATUS y Kaggle se mantienen como fuentes de referencia/complementarias y deben conservar su documentación de acceso.

| Fuente | Archivo o tabla | Filas | Columnas | Formato | Unidad de observación / granularidad inicial | Cobertura observada | Función posible |
|---|---|---:|---:|---|---|---|---|
| SSC | HechosTransito_SSC.csv | 134,079 | 26 | CSV | Hecho de tránsito registrado | 2018-01-01 a 2023-12-31 | **Fuente principal:** frecuencia y severidad registrada |
| SSC | VehiculosInvolucrados_SSC.csv | 215,079 | 26 | CSV | Vehículo asociado a un folio de hecho | 2018-01-01 a 2023-12-31 | Caracterizar vehículos involucrados |
| SSC | PersonasLyFporEdad_SSC.csv | 158,077 | 23 | CSV | Registro de persona/condición por edad asociado a folio; confirmar semántica exacta | 2018-01-01 a 2023-12-31 | Describir lesionados/fallecidos por edad |
| SSC | PersonasLyFporSexo_SSC.csv | 144,044 | 24 | CSV | Registro agregado/asociado a folio por sexo y condición; confirmar documentación | 2018-01-01 a 2023-12-31 | Describir lesionados/fallecidos por sexo |
| SSC | PersonasLyFporTipoPersona_SSC.csv | 107,795 | 24 | CSV | Registro asociado a folio por tipo de persona y condición | 2020-01-01 a 2023-12-31 | Describir peatón/conductor/pasajero u otras categorías disponibles |
| C5 | inViales_2014_2015.csv | 356,072 | 17 | CSV | Incidente vial reportado / folio C5 | 2013-12-31 a 2015-12-31 en fecha_creacion | Contexto temporal y geográfico |
| C5 | inViales_2016_2018.csv | 664,111 | 17 | CSV | Incidente vial reportado / folio C5 | 2015-12-31 a 2018-12-31 | Contexto temporal y geográfico |
| C5 | inViales_2019_2021.csv | 590,636 | 17 | CSV | Incidente vial reportado / folio C5 | 2018-11-15 a 2021-12-31 | Contexto temporal y geográfico |
| C5 | inViales_2022_2024.csv | 504,261 | 17 | CSV | Incidente vial reportado / folio C5 | 2021-12-29 a 2024-02-29 | Contexto temporal y geográfico |
| INEGI | ATUS | Pendiente de incorporar CSV oficial | 46 variables documentadas | CSV/microdatos | Accidente registrado | Nacional; filtrar CDMX posteriormente | Referencia estadística/metodológica |
| Kaggle | accidentes.db | Base SQLite documentada por el equipo | Varias tablas | SQLite | Accidente y catálogos | México, 1997–2022 | Referencia histórica/estructural |

> Las fechas anteriores son rangos observados directamente en las columnas de fecha de los archivos revisados. Que un archivo llamado “2014-2015” contenga un registro de 2013-12-31, por ejemplo, se conserva como **señal visible** y no se corrige en esta etapa.

## 2. Código reproducible opcional

El notebook **no falla si los CSV no están en el repositorio**. Si en algún momento se copian temporalmente los datos a data/raw/, este bloque permite recalcular la evidencia.

La documentación que sigue ya contiene los resultados obtenidos de los archivos revisados.

In [1]:
import pandas as pd
from IPython.display import display

# Resultados obtenidos durante la primera lectura de los datos crudos
inventario = pd.DataFrame([
    ["SSC","HechosTransito_SSC.csv",134079,26,126472,0,"2018-01-01 a 2023-12-31"],
    ["SSC","VehiculosInvolucrados_SSC.csv",215079,26,126472,0,"2018-01-01 a 2023-12-31"],
    ["SSC","PersonasLyFporEdad_SSC.csv",158077,23,126253,3498,"2018-01-01 a 2023-12-31"],
    ["SSC","PersonasLyFporSexo_SSC.csv",144044,24,126472,0,"2018-01-01 a 2023-12-31"],
    ["SSC","PersonasLyFporTipoPersona_SSC.csv",107795,24,99000,0,"2020-01-01 a 2023-12-31"],
    ["C5","inViales_2014_2015.csv",356072,17,356072,0,"2013-12-31 a 2015-12-31"],
    ["C5","inViales_2016_2018.csv",664111,17,664108,0,"2015-12-31 a 2018-12-31"],
    ["C5","inViales_2019_2021.csv",590636,17,590627,0,"2018-11-15 a 2021-12-31"],
    ["C5","inViales_2022_2024.csv",504261,17,504261,0,"2021-12-29 a 2024-02-29"]
], columns=["Fuente","Archivo","Filas","Columnas","Folios distintos","Duplicados completos","Cobertura observada"])

display(inventario)

Fuente,Archivo,Filas,Columnas,Folios distintos,Duplicados completos,Cobertura observada
SSC,HechosTransito_SSC.csv,134079,26,126472,0,2018-01-01 a 2023-12-31
SSC,VehiculosInvolucrados_SSC.csv,215079,26,126472,0,2018-01-01 a 2023-12-31
SSC,PersonasLyFporEdad_SSC.csv,158077,23,126253,3498,2018-01-01 a 2023-12-31
SSC,PersonasLyFporSexo_SSC.csv,144044,24,126472,0,2018-01-01 a 2023-12-31
SSC,PersonasLyFporTipoPersona_SSC.csv,107795,24,99000,0,2020-01-01 a 2023-12-31
C5,inViales_2014_2015.csv,356072,17,356072,0,2013-12-31 a 2015-12-31
C5,inViales_2016_2018.csv,664111,17,664108,0,2015-12-31 a 2018-12-31
C5,inViales_2019_2021.csv,590636,17,590627,0,2018-11-15 a 2021-12-31
C5,inViales_2022_2024.csv,504261,17,504261,0,2021-12-29 a 2024-02-29


## Resultados visibles

Las salidas de las celdas se conservaron en este notebook para que los resultados de la primera lectura sean visibles directamente al abrirlo desde GitHub. Los datos crudos pesados no necesitan estar dentro del repositorio para consultar esta evidencia.


# 3. Primera lectura — SSC

## 3.1 HechosTransito_SSC.csv

**Resultado de la lectura:** 134,079 filas y 26 columnas. Se observaron **126,472 folios distintos** y **0 filas exactamente duplicadas**.

**Unidad de observación:** para el proyecto se interpreta como un **hecho de tránsito registrado por SSC**. El folio funciona como identificador del hecho, pero no debe asumirse como una clave única de todas las tablas auxiliares porque un hecho puede relacionarse con vehículos y personas.

**Cobertura observada:** fecha_evento va de **2018-01-01 a 2023-12-31**.

**Variables directamente útiles para la pregunta:** fecha_evento, hora_evento, tipo_evento, folio, latitud, longitud, colonia, alcaldia, dia, personas_fallecidas y personas_lesionadas.

**Señales visibles:** no hay duplicados completos, pero sí existen valores faltantes. Entre los más numerosos se observaron matricula_unidad_medica (82,187), unidad_a_cargo (65,746), fecha_captura (38,200) y origen (35,043). Esto se registra como evidencia; todavía no se imputan ni eliminan valores.

**Función en el proyecto:** será la **fuente principal**, porque contiene simultáneamente ubicación, fecha/hora y las variables de lesionados y fallecidos necesarias para distinguir frecuencia de severidad registrada.

In [2]:
# Resultados de HechosTransito_SSC.csv
resultados_hechos = pd.DataFrame({
    "Indicador":["Filas","Columnas","Folios distintos","Duplicados completos","Fecha mínima","Fecha máxima"],
    "Resultado":[134079,26,126472,0,"2018-01-01","2023-12-31"]
})
display(resultados_hechos)

Indicador,Resultado
Filas,"134,079"
Columnas,26
Folios distintos,"126,472"
Duplicados completos,0
Fecha mínima,2018-01-01
Fecha máxima,2023-12-31


In [3]:
# Principales valores faltantes encontrados
faltantes_hechos = pd.DataFrame({
    "Columna":["matricula_unidad_medica","unidad_a_cargo","fecha_captura","origen"],
    "Valores faltantes":[82187,65746,38200,35043]
})
display(faltantes_hechos)

Columna,Valores faltantes
matricula_unidad_medica,"82,187"
unidad_a_cargo,"65,746"
fecha_captura,"38,200"
origen,"35,043"


## 3.2 VehiculosInvolucrados_SSC.csv

**Resultado:** 215,079 filas × 26 columnas; **0 duplicados completos**; 126,472 folios distintos.

**Granularidad:** no representa un hecho único. La presencia de no_vehiculo y tipo_vehiculo, junto con un número de filas superior a la tabla de hechos, indica una granularidad relacionada con **vehículos involucrados por folio**.

**Cobertura observada:** 2018-01-01 a 2023-12-31.

**Señales visibles:** matricula_unidad_medica presenta 130,262 faltantes y unidad_a_cargo 104,109, entre otros campos con ausencia.

**Duda inicial:** antes de unir esta tabla con hechos debemos comprobar la relación uno-a-muchos mediante folio; no se puede sumar el número de filas de vehículos como si fueran hechos de tránsito.

## 3.3 Personas lesionadas y fallecidas — tablas auxiliares SSC

### Por edad
PersonasLyFporEdad_SSC.csv: **158,077 × 23**, cobertura 2018-01-01 a 2023-12-31, 126,253 folios distintos y **3,498 duplicados completos aparentes**.

Variables particulares: condicion_persona y edad.

### Por sexo
PersonasLyFporSexo_SSC.csv: **144,044 × 24**, cobertura 2018-01-01 a 2023-12-31, 126,472 folios distintos y **0 duplicados completos**.

Variables particulares: sexo, condicion_persona y total.

### Por tipo de persona
PersonasLyFporTipoPersona_SSC.csv: **107,795 × 24**, cobertura observada 2020-01-01 a 2023-12-31, 99,000 folios distintos y **0 duplicados completos**.

Variables particulares: tipo_persona, condicion_persona y total.

### Interpretación y duda
Estas tablas tienen una granularidad distinta de HechosTransito_SSC.csv. Por ello no se deben concatenar ni sumar directamente. La tabla por edad, además, presenta 3,498 filas exactamente repetidas; en esta etapa **no se eliminan**, porque primero debe determinarse si son duplicados erróneos o registros válidos bajo la semántica de la fuente.

In [4]:
# Resultados de las tablas de personas de SSC
personas_ssc = pd.DataFrame([
    ["Personas por edad",158077,23,126253,3498],
    ["Personas por sexo",144044,24,126472,0],
    ["Personas por tipo de persona",107795,24,99000,0]
], columns=["Tabla","Filas","Columnas","Folios distintos","Duplicados completos"])
display(personas_ssc)

Tabla,Filas,Columnas,Folios distintos,Duplicados completos
Personas por edad,"158,077",23,"126,253","3,498"
Personas por sexo,"144,044",24,"126,472",0
Personas por tipo de persona,"107,795",24,"99,000",0


# 4. Primera lectura — C5

Los cuatro CSV revisados tienen **17 columnas con los mismos nombres**:

folio, fecha_creacion, hora_creacion, dia_semana, fecha_cierre, hora_cierre, tipo_incidente_c4, incidente_c4, alcaldia_inicio, codigo_cierre, clas_con_f_alarma, tipo_entrada, alcaldia_cierre, alcaldia_catalogo, colonia_catalogo, longitud, latitud.

Esto facilita una posible integración longitudinal posterior, pero todavía deben comprobarse categorías, definiciones y reglas de registro entre periodos.

## 4.1 Resultados por archivo C5

| Archivo | Filas × columnas | Folios distintos | Duplicados completos | Principales faltantes observados |
|---|---:|---:|---:|---|
| inViales_2014_2015.csv | 356,072 × 17 | 356,072 | 0 | colonia_catalogo 6,364; alcaldia_catalogo 1,512; alcaldia_inicio 75 |
| inViales_2016_2018.csv | 664,111 × 17 | 664,108 | 0 | colonia_catalogo 12,615; alcaldia_catalogo 1,601; alcaldia_inicio 52 |
| inViales_2019_2021.csv | 590,636 × 17 | 590,627 | 0 | colonia_catalogo 12,462; hora_cierre 1,544; hora_creacion 1,544; alcaldia_catalogo 1,272 |
| inViales_2022_2024.csv | 504,261 × 17 | 504,261 | 0 | colonia_catalogo 11,176; alcaldia_catalogo 554; alcaldia_inicio 35; tipo_entrada 5 |

### Unidad de observación
Cada fila se interpreta inicialmente como un **incidente vial reportado al C5 identificado por un folio**. Esta unidad no debe confundirse con el hecho de tránsito de SSC.

### Función posible
C5 complementará el análisis con contexto temporal y geográfico de incidentes reportados. No se utilizará para afirmar que un mayor número de reportes equivale automáticamente a mayor “riesgo”.

### Dudas visibles
- En 2016–2018 y 2019–2021 hay menos folios distintos que filas, aunque no existen filas completamente duplicadas. Esto indica que ciertos folios aparecen más de una vez y debe revisarse la razón antes de declarar una clave única.
- Los rangos de fecha_creacion incluyen fechas limítrofes fuera del nombre nominal de algunos archivos; se conserva el hallazgo.
- alcaldia_inicio, alcaldia_cierre y alcaldia_catalogo no deben tratarse como equivalentes sin consultar el diccionario.

In [5]:
# Resultados de los cuatro archivos de C5
resultados_c5 = pd.DataFrame([
    ["2014-2015",356072,356072,0,6364],
    ["2016-2018",664111,664108,0,12615],
    ["2019-2021",590636,590627,0,12462],
    ["2022-2024",504261,504261,0,11176]
], columns=["Periodo","Filas","Folios distintos","Duplicados","Faltantes colonia_catalogo"])
display(resultados_c5)

Periodo,Filas,Folios distintos,Duplicados,Faltantes colonia_catalogo
2014-2015,"356,072","356,072",0,"6,364"
2016-2018,"664,111","664,108",0,"12,615"
2019-2021,"590,636","590,627",0,"12,462"
2022-2024,"504,261","504,261",0,"11,176"


# 5. INEGI — ATUS

INEGI-ATUS se conserva como **fuente estadística/metodológica de referencia**. Para la primera lectura final debe incorporarse el CSV oficial cuando se descargue.

**Unidad de observación documentada:** accidente registrado dentro del universo estadístico de ATUS.

**Cobertura:** nacional, con desagregación territorial. Para utilizarlo en el análisis de CDMX será necesario identificar y posteriormente filtrar la entidad correspondiente, pero **ese filtrado no se realiza en esta etapa**.

**Estructura documentada:** 46 variables.

**Uso previsto:** contrastar estructura, definiciones y contexto estadístico. No se inventan conteos de nulos, duplicados ni dimensiones del archivo mientras el CSV oficial no haya sido incorporado y ejecutado.

### Estado
**Aceptada con cautela / pendiente de evidencia de archivo crudo.** La fuente es pertinente, pero el equipo todavía debe conservar el CSV oficial dentro de su evidencia externa o indicar claramente su ruta de descarga.

# 6. Kaggle — Accidentes viales en México 1997–2022

El dataset de Kaggle se documentó como una base SQLite (accidentes.db) con una tabla principal de accidentes y tablas catálogo.

**Unidad de observación:** accidente en la tabla principal; las tablas de entidades/municipios corresponden a catálogos y tienen otra granularidad.

**Cobertura documentada:** México, 1997–2022.

**Uso:** referencia histórica y estructural. No se debe sumar con ATUS como si fueran dos fuentes estadísticas independientes sin comprobar su procedencia, ya que el dataset está relacionado con información de accidentes de México y debe conservarse la trazabilidad de su origen.

### Estado
**Aceptada con cautela.** Sirve como referencia y apoyo analítico, pero la fuente principal para responder la pregunta local continúa siendo SSC.

# 7. Inventario resumido de columnas clave

## SSC — Hechos
- **Identificación:** folio
- **Tiempo:** fecha_evento, hora_evento, dia
- **Geografía:** latitud, longitud, colonia, alcaldia
- **Evento:** tipo_evento
- **Severidad registrada:** personas_lesionadas, personas_fallecidas
- **Contexto vial:** zona_vial, tipo_de_interseccion, interseccion_semaforizada, clasificacion_de_la_vialidad, sentido_de_circulacion

## C5
- **Identificación:** folio
- **Tiempo:** fecha_creacion, hora_creacion, dia_semana, fecha_cierre, hora_cierre
- **Evento:** tipo_incidente_c4, incidente_c4
- **Geografía:** alcaldia_inicio, alcaldia_cierre, alcaldia_catalogo, colonia_catalogo, longitud, latitud
- **Cierre/clasificación:** codigo_cierre, clas_con_f_alarma
- **Origen:** tipo_entrada

El inventario completo de todas las columnas debe conservarse en la evidencia reproducible; aquí se muestran las variables más relevantes para la pregunta de negocio.

# 8. Señales visibles y dudas iniciales

| Señal | Evidencia de primera lectura | Tratamiento en esta etapa |
|---|---|---|
| Valores faltantes | Presentes en SSC y C5; destacan campos de unidad médica, captura, vialidad y catálogos | Se documentan; no se imputan |
| Duplicados completos | 3,498 en SSC personas por edad; 0 en los demás CSV revisados | No se eliminan todavía |
| Folios repetidos | Algunos archivos tienen más filas que folios únicos | Revisar granularidad y relación uno-a-muchos |
| Coberturas distintas | SSC 2018–2023; tipo de persona 2020–2023; C5 dividido desde 2014 nominalmente | No mezclar periodos sin delimitación |
| Fechas limítrofes | Algunos archivos C5 contienen fechas previas al periodo indicado en el nombre | Conservar y revisar posteriormente |
| Campos geográficos múltiples | C5 contiene tres campos de alcaldía | Consultar diccionario antes de elegir |
| Granularidades diferentes | Hechos, vehículos, personas, incidentes y catálogos | No unir sin validar claves |
| Severidad | SSC contiene personas_lesionadas y personas_fallecidas directamente | Utilizar después para indicadores descriptivos |
| Causalidad | Los datos permiten observar patrones, no demostrar causas | Evitar llamar “factores de riesgo” a simples asociaciones |

# 9. Relación con la pregunta de negocio

La primera lectura indica que la pregunta es **viable principalmente con SSC**:

- alcaldia permite agrupar territorialmente;
- hora_evento permite construir bloques horarios posteriormente;
- folio y los registros de hechos permiten medir frecuencia bajo una regla definida;
- personas_lesionadas y personas_fallecidas permiten construir indicadores de **severidad registrada**.

C5 puede complementar el contexto de incidentes reportados, pero no se utilizará como sustituto automático de SSC. INEGI-ATUS y Kaggle sirven como referencia y contraste.

También se mantiene una distinción importante: una alcaldía con más hechos registrados **no se denominará automáticamente “más riesgosa”**, porque la frecuencia observada no incorpora por sí sola exposición, población, flujo vehicular u otros denominadores.

# 10. Conclusión de la primera lectura

Con la evidencia disponible, el equipo cuenta con datos suficientes para continuar con un análisis descriptivo de **frecuencia y severidad registrada de hechos de tránsito por alcaldía y horario**.

La principal decisión metodológica es mantener separadas las fuentes y tablas hasta comprobar sus claves, granularidades y definiciones. Los nulos, duplicados aparentes, folios repetidos, diferencias temporales y códigos pendientes se registran como problemas visibles, pero **no se corrigen en el Avance 1**.

La fuente principal será HechosTransito_SSC.csv. Las tablas auxiliares de SSC permitirán ampliar el análisis de vehículos y personas cuando su granularidad haya sido validada. C5 se utilizará como fuente complementaria; INEGI-ATUS como referencia estadística/metodológica; y Kaggle como referencia histórica/estructural.

Este notebook puede permanecer en GitHub aun cuando los archivos crudos grandes se almacenen fuera del repositorio, ya que conserva los resultados de la primera lectura y el código necesario para reproducirlos cuando los datos estén disponibles.